# Autoship Delay in Onboarding — Power Analysis

**Experiment:** V2 Autoship Delay in Onboarding · **Owner:** Sergio Oyola · **Metric used for power analysis:** First Fix Conversion (cancellation adjusted) · **Randomization unit:** `client_id` · **Allocation point:** Style Profile complete

## Design choices

- **Population**: signed-up clients completing Style Profile, **Womens + Mens** business lines, restricted to **web (desktop + mobile)** only (per doc's Target Population). `curated.client` / `curated.client_first_conversion` have no platform column to filter on directly — see Step 0 for how the web restriction is recovered and validated.

- **Classic 2-arm A/B**: Control (pre-order placement, status quo) vs. Treatment (delayed enrollment + 10% post-checkout nudge). Single comparison, **50/50 split**, `alpha = 0.05` — no Bonferroni correction needed (only one comparison).

- **One-sided test.** Sizing for a one-sided +MDE does **not** symmetrically size for harm, so no Harm/guardrail grid is computed here. Downside risk on this metric is instead caught qualitatively via the Decision Matrix, and margin risk via the 90-day contribution margin guardrail metric (per the doc's Risk section) — neither is powered in this notebook.

- **Metric definition** (per doc): ***First Fix Conversion (cancellation adjusted) = # cancellation-adjusted first fix requests / # Style Profile sign-up complete.***
  - Numerator: `curated.client_first_conversion.cancellation_adjusted_first_fix_demand_ts IS NOT NULL`
  - Denominator: `curated.client.style_profile_completed_ts IS NOT NULL`, restricted to `platform = 'web'` per the Population bullet above — this experiment's actual allocation point.

- **Maturation window**: a cancellation-adjusted first-fix flag needs time to resolve after Style Profile completion (client schedules a Fix -> it ships or gets cancelled). This notebook uses a **60-day maturation buffer** (cohort restricted to `style_profile_completed_ts <= CURRENT_DATE - 60 days`) as a conservative placeholder, chosen by inspection of the monthly series below (rates stabilize by month 2). Not yet confirmed against a table owner/pipeline SLA.

In [ ]:
import numpy as np
import pandas as pd
from amphibian import get_data_accessor
from power import n_total_statsmodels

# Pandas display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

def query(sql):
    return get_data_accessor(engines=['presto']).fetch_sql(sql=sql)

# Data parameters
BUSINESS_LINES = ('Womens', 'Mens')
COHORT_START = '2025-10-01'
MATURATION_DAYS = 60
SP_EVENT_WINDOW_PRE_HOURS = 6   # window around style_profile_completed_ts used to find the matching platform event
SP_EVENT_WINDOW_POST_HOURS = 1

# Design parameters
ALPHA = 0.05
POWER = 0.80
TWO_SIDED = False   # one-sided per Experiment Design doc
N_ARMS = 2
SPLIT = 0.5
MDE_GRID = [0.02, 0.03, 0.05, 0.10]

## Step 0 — Validate the platform join (match rate & platform split)

`curated.client.style_profile_completed_ts` marks when a client finished Style Profile, not what platform they used. Platform is recovered by joining each cohort client to the nearest matching event in `curated.product_tracking_events`, which requires two judgment calls: a time window around the completion timestamp, and a definition of which event types count as Style Profile activity. Both are documented below, then checked against actual match rates.

### How `rn` picks the matching event

There is no one event that unambiguously marks "Style Profile complete" (the flow logs a `save_style_profile` event per question, not once at the end), so no single candidate can be picked as definitively correct. Nearest-in-time is used as the
best available proxy: whichever event is closest to the completion timestamp is the one most likely to reflect the device the client was actually on at that moment, rather than, say, an unrelated session from hours earlier on a different device that happens to also fall inside the window.

For each cohort client, every candidate event found in the window is ranked via `ROW_NUMBER() OVER (PARTITION BY client_id ORDER BY ABS(date_diff('second', event_ts, style_profile_completed_ts)))` — ordered by absolute distance in time from `style_profile_completed_ts`, closest first. Only `rn = 1`, the single nearest candidate, is kept; every other candidate in the window is discarded.

### Logic behind the [-6h, +1h] window

Style Profile is a multi-question flow, and the underlying `save_style_profile` event fires per question rather than once at the end, so there is no single event that unambiguously marks completion. Platform is instead read off the nearest Style-Profile-flow event within a bounded window around `style_profile_completed_ts`:

- **-6 hours (look back)**: wide enough to cover the session in which the questions were actually answered — a single sitting, possibly with a short break — without reaching back so far that it risks picking up an unrelated, earlier visit on a different device.
- **+1 hour (look forward)**: a small buffer for event-logging lag/clock skew, or a trailing confirmation-screen event that logs slightly after the completion timestamp is stamped.

The window is a reasonable assumption based on how the flow behaves, not a value derived from measured session-length data. Confirming 6h/1h are the right boundaries (versus a wider or narrower window) would need either a table owner's input or an empirical look at the gaps between consecutive Style Profile events per client.

### Logic behind the event-type filter

`curated.product_tracking_events` logs every kind of client activity, so the platform lookup is scoped to event types actually tied to the Style Profile flow:

- `schema = 'style_profile_view'`: viewing a question.
- `schema = 'style_profile_select'`: answering/selecting a question, including the final save (`action_name = 'save_style_profile'`) and skip (`action_name = 'skip_question'`) actions.
- `type = 'screen_view'` with `screen_view_name IN ('style_profile', 'client_style_profile', 'stylefile_onboarding')`: landing on the Style Profile screen itself.

Style Profile is served under two different screen names in the tracking data — `style_profile` and `stylefile_onboarding` (an apparently rebranded onboarding variant) — both using the identical `style_profile_select`/`style_profile_view` schema for question-level events. Because the schema-based match isn't scoped to a specific screen name, it already captures both variants without needing to enumerate them; `stylefile_onboarding` is included in the screen-view-only clause for completeness, covering the edge case of a client whose only nearby signal is landing on that screen with no adjacent question event logged (validated below to make zero difference to the match rate in the samples checked).

This restriction reflects the event types actually observed in the data via ad hoc keyword search.

### Match-rate validation

Before trusting a web-only baseline, the join should match most cohort clients to a platform, with a plausible resulting split. Checked on two cohort months at opposite ends of the date range used below — April 2026 (most recent) and October 2025 (oldest) — to also confirm the match rate doesn't degrade for older data.


In [2]:
def validate_platform_match(cohort_start, cohort_end, events_start, events_end):
    sql = f"""--sql
WITH cohort AS (
    SELECT client_id, style_profile_completed_ts, business_line
    FROM curated.client
    WHERE style_profile_completed_ts IS NOT NULL
      AND business_line IN {BUSINESS_LINES}
      AND DATE(style_profile_completed_ts) >= DATE '{cohort_start}'
      AND DATE(style_profile_completed_ts) < DATE '{cohort_end}'
),
sp_events AS (
    SELECT client_id, platform, datetime_in_utc
    FROM curated.product_tracking_events
    WHERE client_id IS NOT NULL
      AND date_in_utc >= DATE '{events_start}'
      AND date_in_utc < DATE '{events_end}'
      AND (
            schema IN ('style_profile_select', 'style_profile_view')
            OR (type = 'screen_view' AND screen_view_name IN ('style_profile', 'client_style_profile', 'stylefile_onboarding'))
          )
),
matched AS (
    SELECT
        c.client_id,
        e.platform,
        ROW_NUMBER() OVER (
            PARTITION BY c.client_id
            ORDER BY ABS(date_diff('second', e.datetime_in_utc, c.style_profile_completed_ts))
        ) AS rn
    FROM cohort c
    LEFT JOIN sp_events e
        ON e.client_id = c.client_id
        AND e.datetime_in_utc BETWEEN c.style_profile_completed_ts - INTERVAL '{SP_EVENT_WINDOW_PRE_HOURS}' HOUR
                                   AND c.style_profile_completed_ts + INTERVAL '{SP_EVENT_WINDOW_POST_HOURS}' HOUR
)
SELECT
    (SELECT count(*) FROM cohort) as n_cohort,
    count(*) FILTER (WHERE rn=1 AND platform IS NOT NULL) as n_matched,
    count(*) FILTER (WHERE rn=1 AND platform='web') as n_web,
    count(*) FILTER (WHERE rn=1 AND platform='iOS') as n_ios,
    count(*) FILTER (WHERE rn=1 AND platform NOT IN ('web','iOS')) as n_other_platform
FROM matched
WHERE rn = 1
"""
    df = query(sql)
    df['match_rate'] = df['n_matched'] / df['n_cohort']
    df['web_share_of_matched'] = df['n_web'] / df['n_matched']
    return df

validation_df_apr = validate_platform_match(
    cohort_start='2026-04-01', cohort_end='2026-05-01',
    events_start='2026-03-31', events_end='2026-05-02',
)
validation_df_apr

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,n_cohort,n_matched,n_web,n_ios,n_other_platform,match_rate,web_share_of_matched
0,139646,139636,127297,12339,0,0.999928,0.911635


In [3]:
validation_df_oct = validate_platform_match(
    cohort_start='2025-10-01', cohort_end='2025-10-08',
    events_start='2025-09-30', events_end='2025-10-09',
)
validation_df_oct

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,n_cohort,n_matched,n_web,n_ios,n_other_platform,match_rate,web_share_of_matched
0,35018,35013,31790,3223,0,0.999857,0.907948


**Both reference points show a high match rate**: April 2026 (most recent) — 99.99% matched (139,636 / 139,646), split 91.2% web / 8.8% iOS. October 2025 (oldest cohort month) — 99.99% matched (35,013 / 35,018), split 90.8% web / 9.2% iOS. In both samples, `n_other_platform = 0` and adding `stylefile_onboarding` to the screen-view-only clause changed the match count by zero — the schema-based question/answer events already accounted for every match on their own.

This supports two things:
* The join does not degrade for older data, and the screen-view-only fallback is not doing meaningful work on its own — it is kept for completeness, not because it is currently needed.
* Neither check independently confirms the `[-6h, +1h]` window or event-type restriction are exactly right — that would need a table owner's sign-off on the Style Profile event taxonomy. (The maturation-window buffer used in Step 1 carries a similar unconfirmed-assumption.)

## Step 1 — First Fix Conversion baseline & Style Profile completions per day

Cohort = clients completing Style Profile (`curated.client.style_profile_completed_ts`) in Womens/Mens, restricted to cohort months where the full maturation window has already elapsed. Each cohort client is joined to their nearest Style-Profile-flow event (per Step 0's method) and restricted to `platform = 'web'`, then cancellation-adjusted first-fix conversion is flagged from `curated.client_first_conversion.cancellation_adjusted_first_fix_demand_ts`, counted only if it falls within the maturation window of Style Profile completion.


In [4]:
baseline_query = f"""--sql
WITH cohort AS (
    SELECT client_id, style_profile_completed_ts, business_line
    FROM curated.client
    WHERE style_profile_completed_ts IS NOT NULL
      AND business_line IN {BUSINESS_LINES}
      AND DATE(style_profile_completed_ts) >= DATE '{COHORT_START}'
      AND style_profile_completed_ts <= CURRENT_DATE - INTERVAL '{MATURATION_DAYS}' DAY
),
sp_events AS (
    SELECT client_id, platform, datetime_in_utc
    FROM curated.product_tracking_events
    WHERE client_id IS NOT NULL
      AND date_in_utc >= DATE '2025-09-30'
      AND date_in_utc < DATE '2026-06-02'
      AND (
            schema IN ('style_profile_select', 'style_profile_view')
            OR (type = 'screen_view' AND screen_view_name IN ('style_profile', 'client_style_profile', 'stylefile_onboarding'))
          )
),
matched AS (
    SELECT
        c.client_id,
        c.style_profile_completed_ts,
        e.platform,
        ROW_NUMBER() OVER (
            PARTITION BY c.client_id
            ORDER BY ABS(date_diff('second', e.datetime_in_utc, c.style_profile_completed_ts))
        ) AS rn
    FROM cohort c
    LEFT JOIN sp_events e
        ON e.client_id = c.client_id
        AND e.datetime_in_utc BETWEEN c.style_profile_completed_ts - INTERVAL '{SP_EVENT_WINDOW_PRE_HOURS}' HOUR
                                   AND c.style_profile_completed_ts + INTERVAL '{SP_EVENT_WINDOW_POST_HOURS}' HOUR
),
web_cohort AS (
    SELECT client_id, style_profile_completed_ts
    FROM matched
    WHERE rn = 1 AND platform = 'web'
),
joined AS (
    SELECT
        w.client_id,
        w.style_profile_completed_ts,
        DATE_TRUNC('month', w.style_profile_completed_ts) AS month,
        CASE WHEN v.cancellation_adjusted_first_fix_demand_ts IS NOT NULL
              AND v.cancellation_adjusted_first_fix_demand_ts <= w.style_profile_completed_ts + INTERVAL '{MATURATION_DAYS}' DAY
             THEN 1 ELSE 0 END AS cancellation_adjusted_first_fix_conversion
    FROM web_cohort w
    LEFT JOIN curated.client_first_conversion v ON w.client_id = v.client_id
),
month_days AS (
    SELECT month, COUNT(DISTINCT DATE(style_profile_completed_ts)) AS days_observed
    FROM joined
    GROUP BY month
)
SELECT
    j.month,
    md.days_observed,
    COUNT(*) AS n_style_profile_completions_web,
    SUM(cancellation_adjusted_first_fix_conversion) AS n_cancellation_adjusted_first_fix,
    CAST(SUM(cancellation_adjusted_first_fix_conversion) AS DOUBLE) / COUNT(*) AS first_fix_conversion_rate,
    ROUND(COUNT(*) / CAST(md.days_observed AS DOUBLE), 1) AS style_profile_completions_per_day
FROM joined j
JOIN month_days md ON j.month = md.month
GROUP BY j.month, md.days_observed
ORDER BY j.month DESC
"""

baseline_df = query(baseline_query)
baseline_df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,month,days_observed,n_style_profile_completions_web,n_cancellation_adjusted_first_fix,first_fix_conversion_rate,style_profile_completions_per_day
0,2026-05-01 00:00:00.000,15,57162,15671,0.274151,3810.8
1,2026-04-01 00:00:00.000,30,127297,34253,0.269079,4243.2
2,2026-03-01 00:00:00.000,31,165990,43484,0.261968,5354.5
3,2026-02-01 00:00:00.000,28,153159,36660,0.239359,5470.0
4,2026-01-01 00:00:00.000,31,167817,38732,0.230799,5413.5
5,2025-12-01 00:00:00.000,31,114499,25052,0.218797,3693.5
6,2025-11-01 00:00:00.000,30,120381,26969,0.224030,4012.7
7,2025-10-01 00:00:00.000,31,138113,33590,0.243207,4455.3


**Reference month = April 2026** (`2026-04-01`): the most recent **complete** calendar month (`days_observed = 30`) with the full 60-day maturation window elapsed as of this run. May 2026 is shown for context only — it has just 15 mature days so far, so its rate isn't yet comparable to a full month.


In [5]:
REFERENCE_MONTH = '2026-04-01'
ref = baseline_df[baseline_df['month'].astype(str).str.startswith(REFERENCE_MONTH)].reset_index(drop=True)

BASELINE_RATE = float(ref['first_fix_conversion_rate'][0])
DAILY_ELIGIBLE = float(ref['style_profile_completions_per_day'][0])

print(f"BASELINE_RATE (web only) = {BASELINE_RATE:.4f}  |  DAILY_ELIGIBLE (web-only Style Profile completions/day) = {DAILY_ELIGIBLE:,.1f}")

BASELINE_RATE (web only) = 0.2691  |  DAILY_ELIGIBLE (web-only Style Profile completions/day) = 4,243.2


## Step 2 — Sample size & duration

`n_total_statsmodels` sizes a single pairwise 50/50 comparison; `n_treatment` is read as the **per-arm** requirement, `total_n = 2 x n_per_arm`, and each arm accrues `DAILY_ELIGIBLE / 2` Style Profile completions (web) per day. One-sided (`TWO_SIDED = False`), per the doc.


In [6]:
def size_table(rel_grid, baseline, daily):
    raw = n_total_statsmodels(
        baseline_rate=baseline, mde_relative=rel_grid, split_ratio=[SPLIT],
        alpha=ALPHA, power=POWER, two_sided=TWO_SIDED,
    )
    df = pd.DataFrame(raw).T.reset_index(drop=True)
    df['rel_effect'] = df['mde_relative'].apply(lambda x: f"{x:+.0%}")
    df['n_per_arm'] = df['n_treatment'].astype(int)
    df['n_total'] = df['n_per_arm'] * N_ARMS
    df['days_required'] = np.ceil(df['n_per_arm'] / (daily / N_ARMS)).astype(int)
    df['weeks_required'] = (df['days_required'] / 7).round(1)
    return df[['rel_effect', 'p_treatment', 'n_per_arm', 'n_total', 'days_required', 'weeks_required']]

sided = 'one-sided' if not TWO_SIDED else 'two-sided'
print(f"--- First Fix Conversion, web only (baseline={BASELINE_RATE:.1%}, alpha={ALPHA}, power={POWER:.0%}, {sided}, 50/50 split) ---")
size_table(MDE_GRID, BASELINE_RATE, DAILY_ELIGIBLE)

--- First Fix Conversion, web only (baseline=26.9%, alpha=0.05, power=80%, one-sided, 50/50 split) ---


,rel_effect,p_treatment,n_per_arm,n_total,days_required,weeks_required
0,+2%,0.274461,84498,168996,40,5.7
1,+3%,0.277152,37670,75340,18,2.6
2,+5%,0.282533,13644,27288,7,1.0
3,+10%,0.295987,3461,6922,2,0.3


## Step 3 — Summary for Experiment Design doc

Headline MDE per the doc is **+2% relative** (its explicit iteration list is `[0.02, 0.03, 0.05, 0.10]`, shown in full in Step 2). Paste this into the doc's Power Analysis section.


In [ ]:
TARGET_REL_MDE = 0.02  # doc's stated primary MDE (2% relative lift on First Fix Conversion)

res = n_total_statsmodels(
    baseline_rate=BASELINE_RATE,
    mde_relative=[TARGET_REL_MDE],
    split_ratio=[SPLIT],
    alpha=ALPHA,
    power=POWER,
    two_sided=TWO_SIDED,
)

n_per_arm = int(list(res.values())[0]['n_treatment'])
duration_days = int(np.ceil(n_per_arm / (DAILY_ELIGIBLE / N_ARMS)))

summary = {
    'Metric Used': 'First Fix Conversion (cancellation adjusted) = cancellation-adjusted first fix requests / Style Profile sign-up complete',
    'Population': 'Web (desktop + mobile) only, Womens + Mens — platform recovered via curated.product_tracking_events (client_id + platform), see Step 0',
    'Baseline Value': f"{BASELINE_RATE:.1%} (April 2026, web only, {MATURATION_DAYS}-day maturation window)",
    'Minimum Detectable Effect': f"+{TARGET_REL_MDE:.0%} relative ({BASELINE_RATE:.3f} -> {BASELINE_RATE*(1+TARGET_REL_MDE):.3f})",
    'One/Two-Sided Test': 'One-sided',
    'Significance Level': f"{ALPHA} (single comparison, no multiple-comparison correction)",
    'Statistical Power': f"{POWER:.0%}",
    'Variant Split %': '50% / 50% (Control / Treatment)',
    'Minimum Samples by Variant': f"{n_per_arm:,}",
    'Minimum Samples total': f"{n_per_arm*N_ARMS:,}",
    'Shortest Duration Required': f"{duration_days} days (~{duration_days/7:.1f} weeks)",
}
pd.Series(summary).to_frame('value')

,value
Metric Used,First Fix Conversion (cancellation adjusted) = cancellation-adjusted first fix requests / Style Profile sign-up complete
Population,"Web (desktop + mobile) only, Womens + Mens — platform recovered via curated.product_tracking_events (client_id + platform), see Step 0"
Baseline Value,"26.9% (April 2026, web only, 60-day maturation window)"
Minimum Detectable Effect,+2% relative (0.269 -> 0.274)
One/Two-Sided Test,One-sided
Significance Level,"0.05 (single comparison, no multiple-comparison correction)"
Statistical Power,80%
Variant Split %,50% / 50% (Control / Treatment)
Minimum Samples by Variant,"84,498"
Minimum Samples total,"168,996"
